In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pyomo.environ as pyo
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

from pull_prices import merged_df_clean, merged_df_spike
from params import nodes, mcp, mdp, e, fee

# Create output folder
BASE_DIR = os.getcwd()
PLOT_DIR = os.path.join(BASE_DIR, "results", "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

print("Plots will be saved in:", PLOT_DIR)

2026-04-21 08:37:09 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_LMP', 'version': 12}, 'params': {'market_run_id': 'DAM', 'node': None, 'grp_type': [None, 'ALL', 'ALL_APNODES']}}
2026-04-21 08:37:09 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6&queryname=PRC_LMP&version=12&market_run_id=DAM&node=TH_SP15_GEN-APND,TH_NP15_GEN-APND,TH_ZP26_GEN-APND&startdatetime=20250101T08:00-0000&enddatetime=20250131T08:00-0000
2026-04-21 08:37:20 - DEBUG - Found 1 files: ['20250101_20250131_PRC_LMP_DAM_20260420_20_07_13_v12.csv']
2026-04-21 08:37:20 - DEBUG - Parsing file: 20250101_20250131_PRC_LMP_DAM_20260420_20_07_13_v12.csv
  0%|          | 0/30 [00:00<?, ?it/s]2026-04-21 08:37:23 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_AS', 'version': 12}, 'params': {'market_run_id': ['DAM', 'HASP'], 'anc_type': ['ALL', 'NR', 'RD', 'RMD', 'RMU', 'RU', 'SR'], 'anc_region': ['ALL

Plots will be saved in: d:\battery-storage-optimization-energy-ancillary\results\plots


In [2]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 32)
        self.tr = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(32, 4, batch_first=True), 2
        )
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        return self.out(self.tr(self.fc(x)))


def train_transformer(df):

    if df is None:
        raise ValueError("train_transformer received None")

    df = df.copy()

    if "SP15" not in df.columns:
        raise KeyError("SP15 column missing")

    # EWMA-based expected behavior
    df["expected_price"] = df["SP15"].ewm(span=24, adjust=False).mean()

    return df

In [3]:
def prepare_timeseries(df):

    # average price across nodes (simple fix)
    df_ts = df.groupby("datetime").agg({
        "SP15": "mean",
        "NonSpin": "mean",
        "RegDown": "mean",
        "RegUp": "mean",
        "Spin": "mean"
    }).reset_index()

    df_ts = df_ts.sort_values("datetime").reset_index(drop=True)

    return df_ts

In [4]:
# =========================
# 🔹 GRAPH (LEARNED)
# =========================
def build_graph(df):

    pivot = df.pivot(index="datetime", columns="node", values="SP15")
    pivot = pivot.ffill()

    A = pivot.corr().values
    np.fill_diagonal(A, 0)

    D = np.diag(A.sum(axis=1))
    L = D - A

    return L


# =========================
# 🔹 ANOMALY DETECTION
# =========================
def compute_anomaly(df):

    df = df.copy()

    X = df[["SP15"]].values

    iso = IsolationForest(contamination=0.05)
    lof = LocalOutlierFactor()

    df["iso"] = (iso.fit_predict(X) == -1).astype(int)
    df["lof"] = (lof.fit_predict(X) == -1).astype(int)

    df["res"] = abs(df["SP15"] - df["expected_price"])

    scaler = MinMaxScaler()
    df[["res"]] = scaler.fit_transform(df[["res"]])

    df["anomaly"] = df["res"] + df["iso"] + df["lof"]
    df["anomaly"] = MinMaxScaler().fit_transform(df[["anomaly"]])

    return df

In [6]:
# =========================
# 🔹 OPTIMIZATION
# =========================
def optimize(df, L, anomaly_on=True):

    model = pyo.ConcreteModel()
    T = len(df)

    model.t = pyo.RangeSet(0, T - 1)
    model.n = pyo.Set(initialize=nodes)

    model.price = pyo.Param(model.t, initialize=lambda m, t: df.loc[t, "SP15"])
    model.anom = pyo.Param(model.t, initialize=lambda m, t: df.loc[t, "anomaly"])

    model.buy = pyo.Var(model.t, model.n, bounds=(0, mcp))
    model.sell = pyo.Var(model.t, model.n, bounds=(0, mdp))
    model.soc = pyo.Var(model.t, bounds=(0, mcp))

    # SOC dynamics
    def soc_rule(m, t):
        if t == 0:
            return m.soc[t] == mcp
        return m.soc[t] == m.soc[t - 1] + \
            sum(m.buy[t, n] for n in m.n) * e - \
            sum(m.sell[t, n] for n in m.n) / e

    model.soc_c = pyo.Constraint(model.t, rule=soc_rule)

    # Objective
    def obj(m):
        profit = 0
        for t in m.t:
            w = 1 + (m.anom[t] if anomaly_on else 0)
            for n in m.n:
                profit += m.sell[t, n] * m.price[t] * w \
                          - m.buy[t, n] * m.price[t] / w
        return profit

    model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

    pyo.SolverFactory("highs").solve(model)

    # Extract SOC
    soc = [pyo.value(model.soc[t]) for t in model.t]

    return pyo.value(model.obj), soc

In [7]:
# =========================
# 🔹 OPTIMIZATION
# =========================
def optimize(df, L, anomaly_on=True):

    model = pyo.ConcreteModel()
    T = len(df)

    model.t = pyo.RangeSet(0, T - 1)
    model.n = pyo.Set(initialize=nodes)

    model.price = pyo.Param(model.t, initialize=lambda m, t: df.loc[t, "SP15"])
    model.anom = pyo.Param(model.t, initialize=lambda m, t: df.loc[t, "anomaly"])

    model.buy = pyo.Var(model.t, model.n, bounds=(0, mcp))
    model.sell = pyo.Var(model.t, model.n, bounds=(0, mdp))
    model.soc = pyo.Var(model.t, bounds=(0, mcp))

    # SOC dynamics
    def soc_rule(m, t):
        if t == 0:
            return m.soc[t] == mcp
        return m.soc[t] == m.soc[t - 1] + \
            sum(m.buy[t, n] for n in m.n) * e - \
            sum(m.sell[t, n] for n in m.n) / e

    model.soc_c = pyo.Constraint(model.t, rule=soc_rule)

    # Objective
    def obj(m):
        profit = 0
        for t in m.t:
            w = 1 + (m.anom[t] if anomaly_on else 0)
            for n in m.n:
                profit += m.sell[t, n] * m.price[t] * w \
                          - m.buy[t, n] * m.price[t] / w
        return profit

    model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

    pyo.SolverFactory("highs").solve(model)

    # Extract SOC
    soc = [pyo.value(model.soc[t]) for t in model.t]

    return pyo.value(model.obj), soc

In [8]:
# =========================
# 🔹 PLOTTING (SAVE ONLY)
# =========================

def plot_price_comparison(df, name):

    plt.figure(figsize=(12,5))
    plt.plot(df["SP15"].values, label="Actual Price")
    plt.plot(df["expected_price"].values, label="Expected Price")

    plt.title("Price vs Expected Price")
    plt.legend()
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_price.png")
    plt.savefig(filepath)
    plt.close()


def plot_soc_comparison(base_soc, anom_soc, name):

    plt.figure(figsize=(12,5))
    plt.plot(base_soc, label="Baseline SOC")
    plt.plot(anom_soc, label="Anomaly SOC")

    plt.title("SOC Comparison")
    plt.legend()
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_soc.png")
    plt.savefig(filepath)
    plt.close()


def plot_volatility(df, name):

    df = df.copy()
    df["volatility"] = df["SP15"].rolling(24).std()

    plt.figure(figsize=(12,4))
    plt.plot(df["volatility"], color="purple")

    plt.title("Volatility")
    plt.grid()

    filepath = os.path.join(PLOT_DIR, f"{name}_volatility.png")
    plt.savefig(filepath)
    plt.close()


def plot_profit(c_base, c_anom, a_base, a_anom):

    labels = ["Clean Base", "Clean Anom", "Attack Base", "Attack Anom"]
    values = [c_base, c_anom, a_base, a_anom]

    plt.figure(figsize=(8,5))
    plt.bar(labels, values)

    plt.title("Profit Comparison")
    plt.grid(axis="y")

    filepath = os.path.join(PLOT_DIR, "profit_comparison.png")
    plt.savefig(filepath)
    plt.close()

In [9]:
def run(df):

    # 🔹 KEEP ORIGINAL for graph
    df_graph = df.copy()

    # 🔹 CREATE TIME SERIES for model
    df_ts = prepare_timeseries(df)

    # 🔹 Model pipeline
    df_ts = train_transformer(df_ts)
    df_ts = compute_anomaly(df_ts)

    # 🔹 Graph from ORIGINAL data
    L = build_graph(df_graph)

    # 🔹 Optimization
    base_profit, base_soc = optimize(df_ts, L, False)
    anom_profit, anom_soc = optimize(df_ts, L, True)

    return df_ts, base_profit, anom_profit, base_soc, anom_soc

In [10]:
print("Checking data...")

print("merged_df_clean type:", type(merged_df_clean))
print("merged_df_spike type:", type(merged_df_spike))

if merged_df_clean is None:
    raise ValueError("merged_df_clean is None")

if merged_df_spike is None:
    raise ValueError("merged_df_spike is None")

print("Clean shape:", merged_df_clean.shape)
print("Spike shape:", merged_df_spike.shape)

print("\nSample data:")
print(merged_df_clean.head())

Checking data...
merged_df_clean type: <class 'pandas.core.frame.DataFrame'>
merged_df_spike type: <class 'pandas.core.frame.DataFrame'>
Clean shape: (2160, 9)
Spike shape: (2160, 9)

Sample data:
             datetime              node      SP15  NonSpin  RegDown  \
0 2025-01-01 08:00:00  TH_NP15_GEN-APND  42.15472      0.5  5.15675   
1 2025-01-01 08:00:00  TH_SP15_GEN-APND  41.35992      0.5  5.15675   
2 2025-01-01 08:00:00  TH_ZP26_GEN-APND  41.20665      0.5  5.15675   
3 2025-01-01 09:00:00  TH_NP15_GEN-APND  42.20734      0.5  4.95046   
4 2025-01-01 09:00:00  TH_SP15_GEN-APND  41.38941      0.5  4.95046   

   Regulation Mileage Down  Regulation Mileage Up   RegUp  Spin  
0                      0.0                    0.0  2.0806   0.5  
1                      0.0                    0.0  2.0806   0.5  
2                      0.0                    0.0  2.0806   0.5  
3                      0.0                    0.0  0.5000   0.5  
4                      0.0                    

In [11]:
# =========================
# 🔹 EXECUTION
# =========================

print("Running CLEAN scenario...")

clean_df, c_base, c_anom, c_soc_base, c_soc_anom = run(merged_df_clean)

print("Running ATTACK scenario...")
attack_df, a_base, a_anom, a_soc_base, a_soc_anom = run(merged_df_spike)

print("\n===== RESULTS =====")
print("Clean:", c_base, c_anom)
print("Attack:", a_base, a_anom)
print("Improvement:", a_anom - a_base)

# Save plots
plot_price_comparison(clean_df, "clean")
plot_volatility(clean_df, "clean")
plot_soc_comparison(c_soc_base, c_soc_anom, "clean")

plot_price_comparison(attack_df, "attack")
plot_volatility(attack_df, "attack")
plot_soc_comparison(a_soc_base, a_soc_anom, "attack")

plot_profit(c_base, c_anom, a_base, a_anom)

print(f"\nPlots saved in: {PLOT_DIR}")

Running CLEAN scenario...
Running ATTACK scenario...

===== RESULTS =====
Clean: 11274.07239033334 38759.67704543031
Attack: 11274.07239033334 34342.62167142901
Improvement: 23068.54928109567

Plots saved in: d:\battery-storage-optimization-energy-ancillary\results\plots


In [14]:
# =========================
# 🔹 OPTIONAL: SAVE RESULTS
# =========================

import os

RESULT_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)


# ---- 1. Save Profit Summary ----
results_df = pd.DataFrame({
    "scenario": ["clean_base", "clean_anom", "attack_base", "attack_anom"],
    "profit": [c_base, c_anom, a_base, a_anom]
})

results_path = os.path.join(RESULT_DIR, "results_summary.csv")
results_df.to_csv(results_path, index=False)


# ---- 2. Save SOC (Clean) ----
soc_clean_df = pd.DataFrame({
    "baseline_soc": c_soc_base,
    "anomaly_soc": c_soc_anom
})

soc_clean_path = os.path.join(RESULT_DIR, "soc_clean.csv")
soc_clean_df.to_csv(soc_clean_path, index=False)


# ---- 3. Save SOC (Attack) ----
soc_attack_df = pd.DataFrame({
    "baseline_soc": a_soc_base,
    "anomaly_soc": a_soc_anom
})

soc_attack_path = os.path.join(RESULT_DIR, "soc_attack.csv")
soc_attack_df.to_csv(soc_attack_path, index=False)


# ---- 4. Save Processed Data (with anomaly + expected) ----
clean_df.to_csv(os.path.join(RESULT_DIR, "clean_processed.csv"), index=False)
attack_df.to_csv(os.path.join(RESULT_DIR, "attack_processed.csv"), index=False)


# ---- 5. Save Volatility ----
clean_df["volatility"] = clean_df["SP15"].rolling(24).std()
attack_df["volatility"] = attack_df["SP15"].rolling(24).std()

clean_df[["datetime", "volatility"]].to_csv(
    os.path.join(RESULT_DIR, "clean_volatility.csv"), index=False
)
attack_df[["datetime", "volatility"]].to_csv(
    os.path.join(RESULT_DIR, "attack_volatility.csv"), index=False
)